In [1]:
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

## Helper Functions

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

## The run_prompt Function

This function takes a test case and merges it with our prompt template:

In [3]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

## The run_test_case Function

This function orchestrates running a single test case and grading the result:

In [4]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }

## The run_eval Function

This function coordinates the entire evaluation process:

In [5]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        results.append(run_test_case(test_case))
    return results

## Running the Evaluation

To execute our evaluation pipeline, we load our dataset and run it through our functions:

In [6]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

This file must exist in the same folder, saved in lesson 11. If it is missing, you get ``FileNotFoundError``, so re-run the generator and save step first. The cell may take about 30 seconds, and ``[*]`` beside it means it is still working.

## Examining the Results

The evaluation returns a structured JSON array where each object represents one test case result:

In [7]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Parser\n\nHere's a Python function that parses S3 bucket URIs:\n\n```python\ndef parse_s3_uri(uri):\n    \"\"\"\n    Parse an AWS S3 bucket URI and return bucket and key components.\n    \n    Args:\n        uri (str): S3 URI in the format 's3://bucket-name/path/to/file'\n        \n    Returns:\n        dict: Dictionary with 'bucket' and 'key' fields\n        \n    Raises:\n        ValueError: If the URI format is invalid\n    \"\"\"\n    if not isinstance(uri, str):\n        raise ValueError(\"URI must be a string\")\n    \n    if not uri.startswith('s3://'):\n        raise ValueError(\"URI must start with 's3://'\")\n    \n    # Remove the 's3://' prefix\n    uri_without_prefix = uri[5:]\n    \n    # Check if there's at least a bucket name\n    if not uri_without_prefix:\n        raise ValueError(\"URI must contain at least a bucket name\")\n    \n    # Split by the first '/' to separate bucket from key\n    parts = uri_without_prefix.split('/', 1)\n

In [8]:
print(results[0]["output"])

# AWS S3 URI Parser

Here's a Python function that parses S3 bucket URIs:

```python
def parse_s3_uri(uri):
    """
    Parse an AWS S3 bucket URI and return bucket and key components.
    
    Args:
        uri (str): S3 URI in the format 's3://bucket-name/path/to/file'
        
    Returns:
        dict: Dictionary with 'bucket' and 'key' fields
        
    Raises:
        ValueError: If the URI format is invalid
    """
    if not isinstance(uri, str):
        raise ValueError("URI must be a string")
    
    if not uri.startswith('s3://'):
        raise ValueError("URI must start with 's3://'")
    
    # Remove the 's3://' prefix
    uri_without_prefix = uri[5:]
    
    # Check if there's at least a bucket name
    if not uri_without_prefix:
        raise ValueError("URI must contain at least a bucket name")
    
    # Split by the first '/' to separate bucket from key
    parts = uri_without_prefix.split('/', 1)
    
    bucket = parts[0]
    key = parts[1] if len(parts) > 1 el